# LSTM for Predicting Mouse Decisions from Steinmetz LFP Data

This notebook loads preprocessed LFP data from the Steinmetz dataset (in the format generated by `steinmetz_NMA.ipynb`), specifically focusing on the VISp brain region during the 500ms to 1000ms time window *post-stimulus onset*. We will then train an LSTM model to predict the mouse's decision (left, right, or no-go).

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

In [12]:
# --- Configuration ---
# IMPORTANT: Adjust this path to the directory containing your 'steinmetz_part*.npz' files
PROCESSED_DATA_DIR = os.getcwd() + '/steinmetz_data/raw_sessions/' # Example: '/path/to/your/steinmetz_npz_files/'

In [13]:
SESSION_INDEX = 0 # Which session to analyze from the loaded data (after aggregating all .npz files)
TARGET_BRAIN_REGION = 'VISp'

# Time window for LFP data, relative to stimulus onset
TIME_WINDOW_START_POST_STIM_MS = 500
TIME_WINDOW_END_POST_STIM_MS = 1000


In [28]:
# LSTM Hyperparameters
N_EPOCHS = 50
BATCH_SIZE = 32
LSTM_UNITS_1 = 64
LSTM_UNITS_2 = 32
DENSE_UNITS = 32
DROPOUT_RATE = 0.3

In [29]:
#Configuration for OSF download ---
try:
   target_dir = PROCESSED_DATA_DIR
except NameError:
   print("PROCESSED_DATA_DIR not defined, defaulting to current working directory for downloads.")
   target_dir = os.getcwd()

os.makedirs(target_dir, exist_ok=True)

# Files for general behavioral data, spikes etc.
fname_parts = []
for j in range(3): # NMA tutorials typically use 3 parts
   fname_parts.append(os.path.join(target_dir, 'steinmetz_part%d.npz' % j))

urls_parts = [
   "https://osf.io/agvxh/download",  # Part 0
   "https://osf.io/uv3mw/download",  # Part 1
   "https://osf.io/ehmw2/download",  # Part 2
]

# LFP specific file
fname_lfp = os.path.join(target_dir, 'steinmetz_lfp.npz')
url_lfp = "https://osf.io/kx3v9/download"

# Combine for easier download loop
all_fnames_to_download = fname_parts + [fname_lfp]
all_urls_to_download = urls_parts + [url_lfp]

print("Starting download process...")
for i in range(len(all_fnames_to_download)):
   fname_current = all_fnames_to_download[i]
   url_current = all_urls_to_download[i]
   if not os.path.isfile(fname_current):
       print(f"Downloading {fname_current} from {url_current}...")
       try:
           r = requests.get(url_current)
           r.raise_for_status() # Raise an exception for HTTP errors
       except requests.ConnectionError:
           print(f"!!! Failed to connect to {url_current} for file {fname_current} !!!")
           print("Please check your internet connection.")
           # Consider whether to raise an error or allow script to continue if a file is missing
           # For now, let's print a warning and continue
           print(f"Warning: Skipping {fname_current} due to download failure.")
           continue # Skip to the next file
       except requests.exceptions.RequestException as e:
           print(f"!!! Failed to download {fname_current} from {url_current}: {e} !!!")
           print(f"Warning: Skipping {fname_current} due to download failure.")
           continue # Skip to the next file
       else:
           with open(fname_current, "wb") as fid:
               fid.write(r.content)
           print(f"Successfully downloaded {fname_current}. Size: {os.path.getsize(fname_current)/(1024*1024):.2f} MB")
   else:
       print(f"File {fname_current} already exists. Skipping download. Size: {os.path.getsize(fname_current)/(1024*1024):.2f} MB")
print("Download process complete.")

Starting download process...
Successfully downloaded /Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data/raw_sessions/steinmetz_part0.npz. Size: 83.49 MB
Successfully downloaded /Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data/raw_sessions/steinmetz_part1.npz. Size: 67.14 MB
Successfully downloaded /Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data/raw_sessions/steinmetz_part2.npz. Size: 61.13 MB
Successfully downloaded /Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data/raw_sessions/steinmetz_lfp.npz. Size: 150.36 MB
Download process complete.


In [32]:
# --- Load Main Data (Spikes, Behavior, etc.) ---
alldat_main_list = []
main_npz_files = sorted([f for f in os.listdir(target_dir) if f.startswith('steinmetz_part') and f.endswith('.npz') and f != 'steinmetz_lfp.npz'])

if not main_npz_files:
   raise FileNotFoundError(f"No 'steinmetz_part*.npz' files (excluding lfp) found in {target_dir}.")

print(f"\nFound {len(main_npz_files)} main .npz files to load from {target_dir}:")
for f_name in main_npz_files:
   print(f"  - {f_name}")
   try:
       data_part = np.load(os.path.join(target_dir, f_name), allow_pickle=True)
       if 'dat' in data_part and isinstance(data_part['dat'], (list, np.ndarray)):
           alldat_main_list.extend(data_part['dat'])
       else:
           print(f"Warning: File {f_name} does not contain a 'dat' key with a list of sessions. Skipping.")
   except Exception as e:
       print(f"Error loading {f_name}: {e}. Skipping this file.")

if not alldat_main_list:
   raise ValueError("No main session data (spikes/behavior) could be loaded.")
print(f"Successfully loaded main data from {len(alldat_main_list)} sessions.")

# --- Load LFP Data ---
lfp_file_path = os.path.join(target_dir, 'steinmetz_lfp.npz')
if not os.path.isfile(lfp_file_path):
   raise FileNotFoundError(f"'steinmetz_lfp.npz' not found in {target_dir}. Please ensure it was downloaded.")

print(f"\nLoading LFP data from: {lfp_file_path}")
try:
   dat_LFP_all_sessions = np.load(lfp_file_path, allow_pickle=True)['dat']
   print(f"Successfully loaded LFP data for {len(dat_LFP_all_sessions)} sessions.")
except Exception as e:
   raise ValueError(f"Error loading 'steinmetz_lfp.npz': {e}")


# --- Select Session and Combine Data ---
if SESSION_INDEX >= len(alldat_main_list) or SESSION_INDEX >= len(dat_LFP_all_sessions):
   raise IndexError(
       f"SESSION_INDEX ({SESSION_INDEX}) is out of bounds. "
       f"Main sessions: {len(alldat_main_list)}, LFP sessions: {len(dat_LFP_all_sessions)}."
   )

session_data_main = alldat_main_list[SESSION_INDEX]
session_data_lfp = dat_LFP_all_sessions[SESSION_INDEX]

print(f"\nUsing session {SESSION_INDEX}:")
print(f"  Mouse (from main data): {session_data_main.get('mouse_name', 'N/A')}, Date: {session_data_main.get('date_exp', 'N/A')}")
# You might want to verify that mouse_name and date_exp match between session_data_main and session_data_lfp if they exist in both

# --- Extract data for the LSTM ---
# Essential keys from LFP data:
required_lfp_keys = ['lfp', 'brain_area_lfp'] # Add others if needed from LFP file like 'trough_to_peak_lfp'
missing_lfp_keys = [key for key in required_lfp_keys if key not in session_data_lfp]
if missing_lfp_keys:
   raise KeyError(f"The selected LFP session (index {SESSION_INDEX}) is missing essential LFP data keys: {missing_lfp_keys}.")

lfp_data_full_trial = session_data_lfp['lfp']       # Shape: (n_channels, n_trials, n_timebins_total_trial)
lfp_brain_areas = session_data_lfp['brain_area_lfp'] # Shape: (n_channels,)

# Essential keys from main data (behavior, stimulus, etc.):
# These were the keys your original LSTM notebook was looking for in the combined data.
# Make sure they come from session_data_main now.
required_main_keys = ['response', 'bin_size', 'stim_onset'] # Add 'contrast_left', 'feedback_type' etc. as needed
missing_main_keys = [key for key in required_main_keys if key not in session_data_main]
if missing_main_keys:
   raise KeyError(f"The selected main session (index {SESSION_INDEX}) is missing essential data keys: {missing_main_keys}.")

responses_orig = session_data_main['response']
dt_s = session_data_main['bin_size']
stim_onset_s = session_data_main['stim_onset']
# Potentially other keys like:
# contrast_left = session_data_main.get('contrast_left')
# contrast_right = session_data_main.get('contrast_right')
# feedback_type = session_data_main.get('feedback_type')
# gocue = session_data_main.get('gocue')
# active_trials = session_data_main.get('active_trials')


print(f"LFP data shape (channels, trials, timebins): {lfp_data_full_trial.shape}")
# If lfp_brain_areas is a list of strings, print its length
if isinstance(lfp_brain_areas, list):
    print(f"LFP brain areas (number of channels): {len(lfp_brain_areas)}")
    # Optionally, convert it to a NumPy array if you need array properties later,
    # though for just printing or simple iteration, a list is fine.
    # lfp_brain_areas_np = np.array(lfp_brain_areas)
    # print(f"LFP brain areas (as np.array) shape: {lfp_brain_areas_np.shape}")
else:
    # If it's already a NumPy array (less likely given the error, but good for robustness)
    print(f"LFP brain areas shape: {lfp_brain_areas.shape}")

# You can also print the content to verify
print(f"LFP brain areas content: {lfp_brain_areas}")
print(f"Responses shape: {responses_orig.shape}")
print(f"Time bin size (dt): {dt_s} s")
print(f"Stimulus onset time in LFP snippet: {stim_onset_s} s")



# ... rest of your preprocessing and LSTM training code ...


Found 3 main .npz files to load from /Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data/raw_sessions/:
  - steinmetz_part0.npz
  - steinmetz_part1.npz
  - steinmetz_part2.npz
Successfully loaded main data from 39 sessions.

Loading LFP data from: /Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data/raw_sessions/steinmetz_lfp.npz
Successfully loaded LFP data for 39 sessions.

Using session 0:
  Mouse (from main data): Cori, Date: 2016-12-14
LFP data shape (channels, trials, timebins): (7, 214, 250)
LFP brain areas (number of channels): 7
LFP brain areas content: ['ACA', 'LS', 'MOs', 'CA3', 'DG', 'SUB', 'VISp']
Responses shape: (214,)
Time bin size (dt): 0.01 s
Stimulus onset time in LFP snippet: 0.5 s


## 1. Load Processed Steinmetz Data

This section loads data from `*.npz` files that are assumed to be preprocessed (e.g., by a script like `steinmetz_NMA.ipynb` from the MouseLand repository). Each `.npz` file should contain a `'dat'` key, which is a list of session dictionaries.

In [10]:
alldat_list = []
if not os.path.isdir(PROCESSED_DATA_DIR):
    raise FileNotFoundError(f"Error: Processed data directory not found: {PROCESSED_DATA_DIR}. \n"
                        f"Please set PROCESSED_DATA_DIR to the correct path containing 'steinmetz_part*.npz' files.")

npz_files = sorted([f for f in os.listdir(PROCESSED_DATA_DIR) if f.startswith('steinmetz_part') and f.endswith('.npz')])

if not npz_files:
    raise FileNotFoundError(f"No 'steinmetz_part*.npz' files found in {PROCESSED_DATA_DIR}. \n"
                        f"Please ensure the files are present and PROCESSED_DATA_DIR is correct.")

print(f"Found {len(npz_files)} .npz files to load from {PROCESSED_DATA_DIR}:")
for f_name in npz_files:
    print(f"  - {f_name}")
    try:
        data_part = np.load(os.path.join(PROCESSED_DATA_DIR, f_name), allow_pickle=True)
        if 'dat' in data_part and isinstance(data_part['dat'], (list, np.ndarray)):
            alldat_list.extend(data_part['dat'])
        else:
            print(f"Warning: File {f_name} does not contain a 'dat' key with a list of sessions, or the format is unexpected. Skipping.")
    except Exception as e:
        print(f"Error loading {f_name}: {e}. Skipping this file.")

if not alldat_list:
    raise ValueError("No session data could be loaded. Please check the .npz files and their structure in PROCESSED_DATA_DIR.")

print(f"\nSuccessfully loaded data from {len(alldat_list)} sessions in total.")

if SESSION_INDEX >= len(alldat_list):
    raise IndexError(f"SESSION_INDEX ({SESSION_INDEX}) is out of bounds for the number of loaded sessions ({len(alldat_list)}). Please choose an index between 0 and {len(alldat_list)-1}.")

session_data = alldat_list[SESSION_INDEX]
print(f"Using session {SESSION_INDEX}: Mouse {session_data.get('mouse_name', 'N/A')}, Date {session_data.get('date_exp', 'N/A')}")

# Verify essential keys exist
required_keys = ['lfp', 'brain_area_lfp', 'response', 'bin_size', 'stim_onset']
missing_keys = [key for key in required_keys if key not in session_data]
if missing_keys:
    raise KeyError(f"The selected session (index {SESSION_INDEX}) is missing essential data keys: {missing_keys}. \n"
                   f"Please ensure the .npz files contain these fields for each session as structured by steinmetz_NMA.ipynb.")

lfp_data_full_trial = session_data['lfp']       # Shape: (n_channels, n_trials, n_timebins_total_trial)
lfp_brain_areas = session_data['brain_area_lfp'] # Shape: (n_channels,)
responses_orig = session_data['response']         # Shape: (n_trials,) with values -1, 0, 1
dt_s = session_data['bin_size']                 # Time bin size in seconds (e.g., 0.01)
stim_onset_s = session_data['stim_onset']         # Stimulus onset time in seconds from start of LFP snippet (e.g., 0.5)

print(f"LFP data shape (channels, trials, timebins): {lfp_data_full_trial.shape}")
print(f"Number of LFP brain areas: {len(lfp_brain_areas)}")
print(f"Number of trials: {responses_orig.shape[0]}")
print(f"LFP time bin size (dt): {dt_s} s")


FileNotFoundError: No 'steinmetz_part*.npz' files found in /Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data/raw_sessions/. 
Please ensure the files are present and PROCESSED_DATA_DIR is correct.

## 2. Data Preprocessing for LSTM

- Select LFP channels from the target brain region (VISp).
- Extract the LFP data for the specified time window post-stimulus onset.
- Normalize the LFP data.
- Reshape LFP data for LSTM input: `(n_trials, n_timesteps_in_window, n_visp_channels)`
- Encode behavioral responses for classification.

In [ ]:
# --- Select VISp LFP Channels ---
visp_channel_indices = np.where(lfp_brain_areas == TARGET_BRAIN_REGION)[0]
if len(visp_channel_indices) == 0:
    available_regions = np.unique(lfp_brain_areas)
    raise ValueError(f"No LFP channels found for brain region '{TARGET_BRAIN_REGION}'. \n"
                     f"Available regions in this session: {available_regions}. \n"
                     f"Please check TARGET_BRAIN_REGION or the data for session {SESSION_INDEX}.")
print(f"Found {len(visp_channel_indices)} LFP channels for {TARGET_BRAIN_REGION}.")
lfp_visp_full_trial = lfp_data_full_trial[visp_channel_indices, :, :] # (n_visp_channels, n_trials, n_timebins_total_trial)

# --- Determine Time Window Indices Post-Stimulus Onset ---
t_start_post_stim_s = TIME_WINDOW_START_POST_STIM_MS / 1000.0
t_end_post_stim_s = TIME_WINDOW_END_POST_STIM_MS / 1000.0

# Calculate start and end times relative to the beginning of the LFP trial data
lfp_window_start_s_abs = stim_onset_s + t_start_post_stim_s
lfp_window_end_s_abs = stim_onset_s + t_end_post_stim_s

# Convert these times to indices
start_idx = int(np.round(lfp_window_start_s_abs / dt_s))
end_idx = int(np.round(lfp_window_end_s_abs / dt_s))

# Validate indices
total_timebins_in_trial = lfp_visp_full_trial.shape[2]
if start_idx < 0 or end_idx > total_timebins_in_trial or start_idx >= end_idx:
    raise ValueError(f"Calculated LFP window indices [{start_idx}:{end_idx}] are invalid for total timebins {total_timebins_in_trial}.\n"
                         f"  LFP snippet duration: {total_timebins_in_trial * dt_s:.2f}s.\n"
                         f"  Stimulus onset (T0): {stim_onset_s:.2f}s (index ~{int(stim_onset_s/dt_s)}).\n"
                         f"  Target window post-stim: {t_start_post_stim_s:.2f}s to {t_end_post_stim_s:.2f}s.\n"
                         f"  Absolute window in LFP: {lfp_window_start_s_abs:.2f}s to {lfp_window_end_s_abs:.2f}s.\n"
                         f"  Check TIME_WINDOW settings, dt ({dt_s:.3f}s), and stim_onset ({stim_onset_s:.2f}s).")

print(f"Extracting LFP from time {lfp_window_start_s_abs:.3f}s to {lfp_window_end_s_abs:.3f}s (indices {start_idx} to {end_idx-1}) relative to LFP data start.")
print(f"This corresponds to {t_start_post_stim_s:.3f}s to {t_end_post_stim_s:.3f}s post-stimulus onset.")
num_timesteps_in_window = end_idx - start_idx
print(f"Number of LFP timebins in window: {num_timesteps_in_window}")

if num_timesteps_in_window <= 0:
    raise ValueError("The calculated time window has zero or negative duration. Please check parameters.")

# Segment LFP data for the window: (n_visp_channels, n_trials, n_timesteps_in_window)
lfp_visp_windowed = lfp_visp_full_trial[:, :, start_idx:end_idx]

# Transpose for LSTM: (n_trials, n_timesteps_in_window, n_visp_channels)
X_data = np.transpose(lfp_visp_windowed, (1, 2, 0))
print(f"Shape of X_data before scaling (trials, timesteps, features): {X_data.shape}")

if X_data.shape[0] == 0 or X_data.shape[1] == 0 or X_data.shape[2] == 0:
    raise ValueError(f"LFP data for LSTM is empty after processing: {X_data.shape}. Check data and parameters.")

# --- Scale LFP Data ---
n_trials, _, n_features = X_data.shape
# Reshape for scaling: each feature (channel) scaled across all trials and timesteps
X_data_reshaped = X_data.reshape(-1, n_features)
scaler = StandardScaler()
X_scaled_reshaped = scaler.fit_transform(X_data_reshaped)
X = X_scaled_reshaped.reshape(n_trials, num_timesteps_in_window, n_features) # Reshape back
print(f"Shape of X after scaling: {X.shape}")

# --- Prepare Labels (Responses) ---
# Map responses: -1 (Left) to 0, 0 (No-go) to 1, 1 (Right) to 2
y_mapped = np.zeros_like(responses_orig, dtype=int)
y_mapped[responses_orig == -1] = 0 # Left
y_mapped[responses_orig == 0]  = 1 # No-go
y_mapped[responses_orig == 1]  = 2 # Right

# One-hot encode labels
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y_mapped.reshape(-1, 1))
print(f"Shape of y (one-hot encoded responses): {y.shape}")
print(f"Original response values: {np.unique(responses_orig)}")
print(f"Mapped response values: {np.unique(y_mapped)}")
print(f"Example responses: Original={responses_orig[0]}, Mapped={y_mapped[0]}, One-hot={y[0]}")

## 3. Split Data into Training and Testing Sets

In [ ]:
if X.shape[0] == 0 or y.shape[0] == 0:
    raise ValueError("Input data X or labels y are empty. Cannot proceed with train/test split.")
if X.shape[0] != y.shape[0]:
    raise ValueError(f"Mismatch in number of samples: X has {X.shape[0]} and y has {y.shape[0]}.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_mapped # Stratify by mapped labels before one-hot
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

## 4. Build and Train LSTM Model

In [ ]:
if X_train.size == 0:
    print("Training data is empty. Skipping model definition and training.")
else:
    input_shape = (X_train.shape[1], X_train.shape[2]) # (n_timesteps, n_features)
    num_classes = y_train.shape[1]

    model = Sequential([
        LSTM(LSTM_UNITS_1, input_shape=input_shape, return_sequences=True, name='lstm_1'),
        BatchNormalization(name='bn_1'),
        Dropout(DROPOUT_RATE, name='dropout_1'),
        LSTM(LSTM_UNITS_2, return_sequences=False, name='lstm_2'),
        BatchNormalization(name='bn_2'),
        Dropout(DROPOUT_RATE, name='dropout_2'),
        Dense(DENSE_UNITS, activation='relu', name='dense_1'),
        Dropout(DROPOUT_RATE, name='dropout_3'),
        Dense(num_classes, activation='softmax', name='output_dense')
    ])

    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    model.summary()

    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

    print("\nTraining the model...")
    history = model.fit(X_train, y_train,
                        epochs=N_EPOCHS,
                        batch_size=BATCH_SIZE,
                        validation_split=0.2, # Use part of training data for validation
                        callbacks=[early_stopping],
                        verbose=1)

    # Plot training history
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

## 5. Evaluate Model Performance

In [ ]:
if 'model' not in globals() or X_test.size == 0:
    print("Skipping model evaluation: Model not trained or test data is empty.")
else:
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTest Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")

    y_pred_proba = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred_proba, axis=1)
    y_true_classes = np.argmax(y_test, axis=1) # Convert one-hot back to class labels

    # Original labels: -1 (Left mapped to 0), 0 (No-go mapped to 1), 1 (Right mapped to 2)
    class_names_report = ['Left (orig -1, mapped 0)', 'No-go (orig 0, mapped 1)', 'Right (orig 1, mapped 2)']
    print("\nClassification Report:")
    print(classification_report(y_true_classes, y_pred_classes, target_names=class_names_report, zero_division=0))

    cm = confusion_matrix(y_true_classes, y_pred_classes)
    try:
        import seaborn as sns
        plt.figure(figsize=(7,6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=class_names_report, yticklabels=class_names_report)
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.show()
    except ImportError:
        print("Seaborn not installed. Skipping confusion matrix heatmap. Confusion Matrix array:")
        print(cm)

## 6. Conclusion

This notebook demonstrated loading preprocessed Steinmetz LFP data, focusing on the VISp region and a specific post-stimulus time window, and then training an LSTM model to predict mouse decisions. The model's performance on the test set gives an initial indication of its predictive capability. 

Potential next steps for improvement:
- Hyperparameter tuning (LSTM units, dropout rates, batch size, learning rate).
- Trying different LFP preprocessing techniques (e.g., different frequency bands, power spectral density features).
- Exploring different model architectures (e.g., CNN-LSTM, attention mechanisms).
- Analyzing data from more brain regions or combining data from multiple sessions/mice.
- More sophisticated handling of class imbalance if present.